# Student Performance Analysis
**Name:** Abu Huraira  
**Roll No:** Sp24-BAI-002  
**Course:** AIC270 - Programming for Artificial Intelligence  
**Instructor:** Mr. Shahrukh Naeem  
**University:** COMSATS University Islamabad, Lahore Campus

---
## Common Setup – Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('data/StudentsPerformance.csv')
print('Dataset loaded successfully!')
print('Shape:', df.shape)

---
## Task 1 – Dataset Basic Information

In [ ]:
# First 5 rows
df.head()

In [ ]:
# Dataset info
df.info()

In [ ]:
# Column names
print('Columns:', df.columns.tolist())
print('Total Rows:', len(df))
print('Total Columns:', len(df.columns))

---
## Task 2 – Descriptive Statistics

In [ ]:
# Mean, Median, Mode, Std, Min, Max for all 3 score columns
score_cols = ['math score', 'reading score', 'writing score']

stats = pd.DataFrame({
    'Mean'  : df[score_cols].mean(),
    'Median': df[score_cols].median(),
    'Mode'  : df[score_cols].mode().iloc[0],
    'Std'   : df[score_cols].std(),
    'Min'   : df[score_cols].min(),
    'Max'   : df[score_cols].max()
})

print('=== Descriptive Statistics ===')
print(stats.round(2))

In [ ]:
# Mean vs Median Comparison
print('=== Mean vs Median Comparison ===')
for col in score_cols:
    mean   = df[col].mean()
    median = df[col].median()
    diff   = mean - median
    print(f'\n{col}:')
    print(f'  Mean   = {mean:.2f}')
    print(f'  Median = {median:.2f}')
    print(f'  Difference = {diff:.2f} → {"slightly right skewed" if diff > 0 else "slightly left skewed"}')

In [ ]:
# Average scores by Gender
print('=== Average Scores by Gender ===')
print(df.groupby('gender')[score_cols].mean().round(2))

In [ ]:
# Average scores by Parental Level of Education
print('=== Average Scores by Parental Level of Education ===')
print(df.groupby('parental level of education')[score_cols].mean().round(2))

In [ ]:
# 5 Important Observations
print("""
=== 5 Important Observations ===

1. Female students score higher in reading and writing, while male students score higher in math.

2. Students whose parents have a master's or bachelor's degree tend to score higher across all subjects.

3. The mean and median are very close for all three scores, meaning the data is fairly balanced (not heavily skewed).

4. Writing and reading scores are closely related — students who read well also write well.

5. Math scores have the most variation (highest std), meaning student performance in math is more spread out compared to reading and writing.
""")

---
## Task 3 – Data Cleaning

In [ ]:
# Step 1: Create messy dataset
import random

messy = df.copy()

# Add missing values randomly
for col in score_cols:
    messy.loc[messy.sample(frac=0.05, random_state=42).index, col] = np.nan

# Add duplicate rows
messy = pd.concat([messy, messy.sample(20, random_state=42)], ignore_index=True)

# Add outliers
messy.loc[0, 'math score']    = 200
messy.loc[1, 'reading score'] = -10
messy.loc[2, 'writing score'] = 999

# Save messy dataset
messy.to_csv('data/messy_students.csv', index=False)
print('Messy dataset created and saved!')
print('Shape:', messy.shape)

In [ ]:
# Step 2: Load messy dataset and show problems
messy_df = pd.read_csv('data/messy_students.csv')

print('=== Messy Dataset Info ===')
messy_df.info()

In [ ]:
print('=== Missing Values ===')
print(messy_df.isnull().sum())

print('\n=== Duplicate Rows ===')
print('Number of duplicates:', messy_df.duplicated().sum())

print('\n=== Outliers (scores outside 0-100) ===')
for col in score_cols:
    outliers = messy_df[(messy_df[col] < 0) | (messy_df[col] > 100)]
    print(f'{col}: {len(outliers)} outlier(s)')

In [ ]:
# Step 3: Clean the data
cleaned_df = messy_df.copy()

# Fix outliers — replace values outside 0-100 with NaN
for col in score_cols:
    cleaned_df.loc[(cleaned_df[col] < 0) | (cleaned_df[col] > 100), col] = np.nan
print('Outliers fixed.')

# Fill missing values with median
for col in score_cols:
    median_val = cleaned_df[col].median()
    cleaned_df[col].fillna(median_val, inplace=True)
print('Missing values filled with median.')

# Remove duplicates
cleaned_df.drop_duplicates(inplace=True)
cleaned_df.reset_index(drop=True, inplace=True)
print('Duplicates removed.')

In [ ]:
# Step 4: Save cleaned file and show before vs after
cleaned_df.to_csv('data/cleaned_students.csv', index=False)

print('=== Before vs After Cleaning ===')
print(f'Rows    - Before: {len(messy_df)}  |  After: {len(cleaned_df)}')
print(f'Missing - Before: {messy_df.isnull().sum().sum()}  |  After: {cleaned_df.isnull().sum().sum()}')
print(f'Duplicates - Before: {messy_df.duplicated().sum()}  |  After: {cleaned_df.duplicated().sum()}')
print('\nCleaned dataset saved to data/cleaned_students.csv')

---
## Task 4 – Exploratory Data Analysis (EDA) & Correlation

In [ ]:
# Load cleaned dataset
eda_df = pd.read_csv('data/cleaned_students.csv')
print('Cleaned dataset loaded. Shape:', eda_df.shape)

In [ ]:
# GroupBy: Gender
print('=== Average Scores by Gender ===')
print(eda_df.groupby('gender')[score_cols].mean().round(2))

In [ ]:
# GroupBy: Race/Ethnicity
print('=== Average Scores by Race/Ethnicity ===')
print(eda_df.groupby('race/ethnicity')[score_cols].mean().round(2))

In [ ]:
# GroupBy: Parental Level of Education
print('=== Average Scores by Parental Level of Education ===')
print(eda_df.groupby('parental level of education')[score_cols].mean().round(2))

In [ ]:
# GroupBy: Test Preparation Course
print('=== Average Scores by Test Preparation Course ===')
print(eda_df.groupby('test preparation course')[score_cols].mean().round(2))

In [ ]:
# Correlation Matrix
print('=== Correlation Matrix ===')
corr_matrix = eda_df[score_cols].corr()
print(corr_matrix.round(2))

In [ ]:
# Seaborn Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Correlation Heatmap of Student Scores')
plt.tight_layout()
plt.savefig('data/correlation_heatmap.png')
plt.show()
print('Heatmap saved.')

In [ ]:
# Written Insights
print("""
=== Detailed Insights (EDA) ===

1. GENDER DIFFERENCES:
   Female students consistently outperform male students in reading and writing scores.
   However, male students tend to score slightly higher in math. This pattern suggests
   a gender-based difference in subject preference or teaching approach.

2. PARENTAL EDUCATION IMPACT:
   Students whose parents hold a master's degree or bachelor's degree score noticeably
   higher in all three subjects compared to students whose parents only completed
   high school or some high school. This highlights the strong influence of family
   educational background on student performance.

3. TEST PREPARATION COURSE:
   Students who completed the test preparation course scored significantly higher
   across all subjects compared to those who did not. This confirms that structured
   preparation directly improves academic performance.

4. RACE/ETHNICITY:
   Group E students have the highest average scores across all subjects, while
   Group A students tend to score the lowest. This could reflect differences in
   access to resources, support systems, or socioeconomic factors.

5. CORRELATION BETWEEN SCORES:
   Reading and writing scores are very strongly correlated (close to 1.0), meaning
   students who are good at reading are almost always good at writing too. Math shows
   a moderate correlation with both reading and writing, suggesting these skills
   are related but more independent.

6. LUNCH TYPE:
   Students with standard lunch (likely indicating better socioeconomic status) score
   higher on average than students with free/reduced lunch. This suggests that
   economic factors play a role in academic outcomes.
""")

---
## Task 5 – Data Visualization

In [ ]:
# Plot 1: Histogram of math score with KDE
plt.figure(figsize=(8, 5))
sns.histplot(eda_df['math score'], kde=True, color='steelblue', bins=20)
plt.title('Distribution of Math Scores')
plt.xlabel('Math Score')
plt.ylabel('Number of Students')
plt.grid(True)
plt.tight_layout()
plt.savefig('data/plot1_histogram.png')
plt.show()

In [ ]:
# Plot 2: Boxplot of math score by gender
plt.figure(figsize=(8, 5))
sns.boxplot(x='gender', y='math score', data=eda_df, palette='Set2')
plt.title('Math Score by Gender')
plt.xlabel('Gender')
plt.ylabel('Math Score')
plt.grid(True)
plt.tight_layout()
plt.savefig('data/plot2_boxplot.png')
plt.show()

In [ ]:
# Plot 3: Scatter plot – reading vs writing score colored by gender
plt.figure(figsize=(8, 5))
colors = {'male': 'steelblue', 'female': 'salmon'}
for gender, group in eda_df.groupby('gender'):
    plt.scatter(group['reading score'], group['writing score'],
                label=gender, alpha=0.6, color=colors[gender])
plt.title('Reading Score vs Writing Score by Gender')
plt.xlabel('Reading Score')
plt.ylabel('Writing Score')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('data/plot3_scatter.png')
plt.show()

In [ ]:
# Plot 4: Bar chart – average math score by parental education
avg_math = eda_df.groupby('parental level of education')['math score'].mean().sort_values()

plt.figure(figsize=(10, 5))
avg_math.plot(kind='bar', color='coral', edgecolor='black')
plt.title('Average Math Score by Parental Level of Education')
plt.xlabel('Parental Level of Education')
plt.ylabel('Average Math Score')
plt.xticks(rotation=45, ha='right')
plt.grid(True, axis='y')
plt.tight_layout()
plt.savefig('data/plot4_barchart.png')
plt.show()

In [ ]:
# Plot 5: Combined 2x2 subplot figure
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Student Performance – Combined Visualization', fontsize=16, fontweight='bold')

# Top-left: Histogram
sns.histplot(eda_df['math score'], kde=True, color='steelblue', bins=20, ax=axes[0, 0])
axes[0, 0].set_title('Distribution of Math Scores')
axes[0, 0].set_xlabel('Math Score')
axes[0, 0].set_ylabel('Count')
axes[0, 0].grid(True)

# Top-right: Boxplot
sns.boxplot(x='gender', y='math score', data=eda_df, palette='Set2', ax=axes[0, 1])
axes[0, 1].set_title('Math Score by Gender')
axes[0, 1].set_xlabel('Gender')
axes[0, 1].set_ylabel('Math Score')
axes[0, 1].grid(True)

# Bottom-left: Scatter plot
for gender, group in eda_df.groupby('gender'):
    axes[1, 0].scatter(group['reading score'], group['writing score'],
                       label=gender, alpha=0.5, color=colors[gender])
axes[1, 0].set_title('Reading vs Writing Score by Gender')
axes[1, 0].set_xlabel('Reading Score')
axes[1, 0].set_ylabel('Writing Score')
axes[1, 0].legend()
axes[1, 0].grid(True)

# Bottom-right: Bar chart
avg_math.plot(kind='bar', color='coral', edgecolor='black', ax=axes[1, 1])
axes[1, 1].set_title('Avg Math Score by Parental Education')
axes[1, 1].set_xlabel('Parental Education')
axes[1, 1].set_ylabel('Average Math Score')
axes[1, 1].tick_params(axis='x', rotation=45)
axes[1, 1].grid(True, axis='y')

plt.tight_layout()
plt.savefig('data/best_visualization.png', dpi=150)
plt.show()
print('Combined figure saved as best_visualization.png')